# Homework 2: full WikiArt conditional GAN

This notebook trains on **every decodable painting** from the assigned
[steubk/wikiart](https://www.kaggle.com/datasets/steubk/wikiart) upload.
The previous 2,390-image experiment remains a separate baseline. This run
uses all 27 style folders, 128×128 images, a style-conditional generator and
projection discriminator, spectral normalization, hinge loss, and an EMA
generator. Image quality is an experimental outcome, not guaranteed.

**Before running:** sign into Kaggle, attach `steubk/wikiart` with **Add Input**,
select a **GPU** in Session options, and enable **Internet** for Ignite and the
pretrained Inception-v3 evaluation model. Run cells in order. Use **Save
Version → Save & Run All** to preserve `/kaggle/working` outputs. Full training
may need more than one Kaggle session; checkpoints are saved every epoch.

Prepared images go to `/kaggle/temp`, while checkpoints and evidence go to
`/kaggle/working/full-wikiart-gan`. The source dataset remains attached read-only.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

PROJECT = Path('/kaggle/working/full-wikiart-gan')
DATA = Path('/kaggle/temp/full-wikiart-128')
RUN = PROJECT / 'run'
PROJECT.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Attached input folders:', [str(p) for p in Path('/kaggle/input').iterdir()])
print('Project:', PROJECT)


## Write the training and evaluation scripts

In [ ]:
%%writefile /kaggle/working/full-wikiart-gan/model.py
"""Efficient style-conditional DCGAN for 64x64 or 128x128 paintings."""

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils import spectral_norm


class ConditionalBatchNorm(nn.Module):
    def __init__(self, channels: int, classes: int):
        super().__init__()
        self.norm = nn.BatchNorm2d(channels, affine=False)
        self.affine = nn.Embedding(classes, channels * 2)
        nn.init.zeros_(self.affine.weight)

    def forward(self, x: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        gamma, beta = self.affine(labels).chunk(2, dim=1)
        return self.norm(x) * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]


class GeneratorStage(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, classes: int,
                 first: bool = False):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, out_channels, 4,
                                       stride=1 if first else 2,
                                       padding=0 if first else 1, bias=False)
        self.norm = ConditionalBatchNorm(out_channels, classes)

    def forward(self, x: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        return F.relu(self.norm(self.conv(x), labels))


class Generator(nn.Module):
    def __init__(self, classes: int, image_size: int = 128,
                 latent_dim: int = 128, base_channels: int = 32):
        super().__init__()
        if image_size not in {64, 128}:
            raise ValueError("image_size must be 64 or 128")
        self.classes = classes
        self.image_size = image_size
        self.latent_dim = latent_dim
        self.base_channels = base_channels
        channels = ([base_channels * 8, base_channels * 8, base_channels * 4,
                     base_channels * 2, base_channels]
                    if image_size == 128 else
                    [base_channels * 8, base_channels * 4, base_channels * 2,
                     base_channels])
        self.stages = nn.ModuleList()
        previous = latent_dim
        for index, width in enumerate(channels):
            self.stages.append(GeneratorStage(previous, width, classes, first=index == 0))
            previous = width
        self.to_rgb = nn.ConvTranspose2d(previous, 3, 4, stride=2, padding=1)

    def forward(self, noise: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        x = noise.reshape(noise.size(0), self.latent_dim, 1, 1)
        for stage in self.stages:
            x = stage(x, labels)
        return torch.tanh(self.to_rgb(x))


class Discriminator(nn.Module):
    def __init__(self, classes: int, image_size: int = 128, base_channels: int = 32):
        super().__init__()
        if image_size not in {64, 128}:
            raise ValueError("image_size must be 64 or 128")
        channels = ([3, base_channels, base_channels * 2, base_channels * 4,
                     base_channels * 8, base_channels * 8]
                    if image_size == 128 else
                    [3, base_channels, base_channels * 2, base_channels * 4,
                     base_channels * 8])
        self.layers = nn.ModuleList(spectral_norm(nn.Conv2d(a, b, 4, stride=2, padding=1))
                                    for a, b in zip(channels[:-1], channels[1:]))
        self.linear = spectral_norm(nn.Linear(channels[-1], 1))
        self.class_embedding = nn.Embedding(classes, channels[-1])

    def forward(self, images: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        x = images
        for layer in self.layers:
            x = F.leaky_relu(layer(x), negative_slope=0.2)
        x = x.mean(dim=(2, 3))
        return self.linear(x).flatten() + (self.class_embedding(labels) * x).sum(dim=1)


def initialize(module: nn.Module) -> None:
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        parameter = getattr(module, "weight_orig", module.weight)
        nn.init.normal_(parameter, 0.0, 0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        if bool(module.weight.detach().abs().sum()):
            nn.init.normal_(module.weight, 0.0, 0.02)


In [ ]:
%%writefile /kaggle/working/full-wikiart-gan/prepare.py
"""Prepare every decodable painting from the assigned steubk/wikiart dataset."""

import argparse
import json
from collections import Counter
from pathlib import Path

from PIL import Image, ImageOps


EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
SOURCE_URL = "https://www.kaggle.com/datasets/steubk/wikiart"


def locate_source(input_root: Path) -> Path:
    matches = [p for p in input_root.rglob("Abstract_Expressionism") if p.is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one mounted {SOURCE_URL} dataset below {input_root}; "
            f"found {len(matches)} Abstract_Expressionism directories."
        )
    return matches[0].parent


def prepare(input_root: Path, output: Path, size: int = 128) -> dict:
    source_root = locate_source(input_root)
    candidates = sorted(p for p in source_root.rglob("*")
                        if p.is_file() and p.suffix.lower() in EXTENSIONS)
    if not candidates:
        raise RuntimeError(f"No images found under {source_root}")
    output.mkdir(parents=True, exist_ok=True)
    manifest = {"source_url": SOURCE_URL, "mounted_root": str(source_root),
                "available_images": len(candidates), "image_size": size,
                "prepared_images": 0, "styles": {}, "files": [], "skipped": []}
    style_counts = Counter()
    for index, source in enumerate(candidates):
        relative = source.relative_to(source_root)
        style = relative.parts[0]
        destination = output / style / f"wikiart_{index:06d}.jpg"
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            if not destination.is_file() or destination.stat().st_size == 0:
                with Image.open(source) as image:
                    image = ImageOps.exif_transpose(image).convert("RGB")
                    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS)
                    image.save(destination, "JPEG", quality=90, subsampling=0)
        except (OSError, ValueError) as error:
            manifest["skipped"].append({"source": relative.as_posix(), "error": str(error)})
            continue
        style_counts[style] += 1
        manifest["files"].append({"source": relative.as_posix(),
                                  "prepared": destination.relative_to(output).as_posix()})
        manifest["prepared_images"] += 1
        if manifest["prepared_images"] % 5000 == 0:
            print(f"Prepared {manifest['prepared_images']}/{len(candidates)} paintings", flush=True)
    manifest["styles"] = dict(sorted(style_counts.items()))
    (output / "full_provenance.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Prepared all {manifest['prepared_images']} decodable paintings across "
          f"{len(style_counts)} styles; skipped {len(manifest['skipped'])}.", flush=True)
    return manifest


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--size", type=int, default=128)
    args = parser.parse_args()
    if args.size not in {64, 128}:
        parser.error("--size must be 64 or 128")
    prepare(args.input_root, args.output, args.size)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/full-wikiart-gan/train.py
"""Train a full-dataset, style-conditional painting GAN with EMA checkpoints."""

import argparse
import copy
import csv
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import save_image

from model import Discriminator, Generator, initialize


def make_dataset(root: Path, seed: int, image_size: int,
                 holdout_fraction: float = 0.1, min_images: int = 50000):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3),
    ])
    full = datasets.ImageFolder(root, transform=transform)
    if len(full) < min_images:
        raise ValueError(f"Expected the full WikiArt dataset, found only {len(full)} images")
    order = np.random.default_rng(seed).permutation(len(full))
    holdout_size = round(len(full) * holdout_fraction)
    return full, Subset(full, order[holdout_size:].tolist()), Subset(full, order[:holdout_size].tolist())


def translate(images: torch.Tensor, pixels: int = 2) -> torch.Tensor:
    if pixels <= 0:
        return images
    dy = random.randint(-pixels, pixels)
    dx = random.randint(-pixels, pixels)
    padded = F.pad(images, (pixels, pixels, pixels, pixels), mode="reflect")
    top, left = pixels + dy, pixels + dx
    return padded[:, :, top:top + images.size(2), left:left + images.size(3)]


def update_ema(ema: nn.Module, current: nn.Module, decay: float) -> None:
    with torch.no_grad():
        for target, source in zip(ema.parameters(), current.parameters()):
            target.lerp_(source, 1 - decay)
        for target, source in zip(ema.buffers(), current.buffers()):
            target.copy_(source)


def save_grid(generator: nn.Module, noise: torch.Tensor, labels: torch.Tensor,
              destination: Path) -> None:
    generator.eval()
    with torch.inference_mode():
        generated = generator(noise, labels).float().cpu()
    save_image(generated, destination, nrow=8, normalize=True, value_range=(-1, 1))


def save_checkpoint(path: Path, generator: nn.Module, discriminator: nn.Module,
                    ema: nn.Module, optim_g, optim_d, scaler, epoch: int,
                    step: int, config: dict) -> None:
    state = {"generator": generator.state_dict(), "discriminator": discriminator.state_dict(),
             "ema": ema.state_dict(), "optim_g": optim_g.state_dict(),
             "optim_d": optim_d.state_dict(), "scaler": scaler.state_dict(),
             "epoch": epoch, "step": step, "config": config}
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--image-size", type=int, default=128, choices=[64, 128])
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--latent-dim", type=int, default=128)
    parser.add_argument("--base-channels", type=int, default=32)
    parser.add_argument("--lr-g", type=float, default=0.0001)
    parser.add_argument("--lr-d", type=float, default=0.0002)
    parser.add_argument("--ema-decay", type=float, default=0.999)
    parser.add_argument("--seed", type=int, default=2026)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--resume", type=Path)
    parser.add_argument("--max-hours", type=float, default=10.5)
    parser.add_argument("--max-steps", type=int, default=0, help="Short verification run only")
    parser.add_argument("--multi-gpu", action="store_true")
    parser.add_argument("--allow-small-data", action="store_true", help="Verification run only")
    args = parser.parse_args()
    args.output.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Enable a Kaggle GPU accelerator.")
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda")
    full, train, holdout = make_dataset(args.data, args.seed, args.image_size,
                                        min_images=100 if args.allow_small_data else 50000)
    loader = DataLoader(train, batch_size=args.batch_size, shuffle=True,
                        num_workers=args.workers, pin_memory=True,
                        drop_last=True, persistent_workers=args.workers > 0)
    generator = Generator(len(full.classes), args.image_size, args.latent_dim,
                          args.base_channels).to(device)
    discriminator = Discriminator(len(full.classes), args.image_size,
                                  args.base_channels).to(device)
    generator.apply(initialize)
    discriminator.apply(initialize)
    ema = copy.deepcopy(generator).eval().requires_grad_(False)
    optim_g = torch.optim.Adam(generator.parameters(), lr=args.lr_g, betas=(0.0, 0.9))
    optim_d = torch.optim.Adam(discriminator.parameters(), lr=args.lr_d, betas=(0.0, 0.9))
    scaler = torch.amp.GradScaler("cuda")
    config = {key: str(value) if isinstance(value, Path) else value
              for key, value in vars(args).items()}
    config.update(dataset_size=len(full), train_size=len(train), holdout_size=len(holdout),
                  classes=full.classes, torch_version=str(torch.__version__),
                  gpu_names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    start_epoch, step = 1, 0
    if args.resume:
        state = torch.load(args.resume, map_location="cpu", weights_only=True)
        previous = state["config"]
        for key in ("classes", "dataset_size", "image_size", "latent_dim", "base_channels", "seed"):
            if previous[key] != config[key]:
                raise ValueError(f"Cannot resume: {key} changed")
        generator.load_state_dict(state["generator"])
        discriminator.load_state_dict(state["discriminator"])
        ema.load_state_dict(state["ema"])
        optim_g.load_state_dict(state["optim_g"])
        optim_d.load_state_dict(state["optim_d"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, step = state["epoch"] + 1, state["step"]
        print(f"Resumed after epoch {state['epoch']} step {step}", flush=True)
    (args.output / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    forward_g = (nn.DataParallel(generator) if args.multi_gpu and torch.cuda.device_count() > 1
                 else generator)
    forward_d = (nn.DataParallel(discriminator) if args.multi_gpu and torch.cuda.device_count() > 1
                 else discriminator)
    fixed_noise = torch.randn(64, args.latent_dim, device=device)
    fixed_labels = torch.linspace(0, len(full.classes) - 1, 8, device=device).long().repeat_interleave(8)
    if start_epoch == 1:
        real_grid = torch.stack([train[i][0] for i in range(64)])
        save_image(real_grid, args.output / "real_grid.png", nrow=8,
                   normalize=True, value_range=(-1, 1))
    loss_path = args.output / "losses.csv"
    with loss_path.open("a" if start_epoch > 1 else "w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        if start_epoch == 1:
            writer.writerow(["epoch", "step", "loss_d", "loss_g"])
        begin = time.time()
        for epoch in range(start_epoch, args.epochs + 1):
            torch.manual_seed(args.seed + epoch)
            torch.cuda.manual_seed_all(args.seed + epoch)
            generator.train()
            discriminator.train()
            d_total = g_total = batches = 0
            for real, labels in loader:
                real = real.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                count = real.size(0)
                optim_d.zero_grad(set_to_none=True)
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                    fake = forward_g(torch.randn(count, args.latent_dim, device=device), labels)
                with torch.autocast("cuda", dtype=torch.float16):
                    real_scores = forward_d(translate(real), labels)
                    fake_scores = forward_d(translate(fake.detach()), labels)
                    loss_d = F.relu(1 - real_scores).mean() + F.relu(1 + fake_scores).mean()
                scaler.scale(loss_d).backward()
                scaler.step(optim_d)

                for parameter in discriminator.parameters():
                    parameter.requires_grad_(False)
                optim_g.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    fake = forward_g(torch.randn(count, args.latent_dim, device=device), labels)
                    loss_g = -forward_d(translate(fake), labels).mean()
                scaler.scale(loss_g).backward()
                scaler.step(optim_g)
                scaler.update()
                for parameter in discriminator.parameters():
                    parameter.requires_grad_(True)
                update_ema(ema, generator, args.ema_decay)
                step += 1
                batches += 1
                d_total += loss_d.item()
                g_total += loss_g.item()
                writer.writerow([epoch, step, loss_d.item(), loss_g.item()])
                if step % 100 == 0:
                    print(f"epoch {epoch}/{args.epochs} step {step}: "
                          f"D={loss_d.item():.3f} G={loss_g.item():.3f}", flush=True)
                if args.max_steps and step >= args.max_steps:
                    break
            if batches == 0:
                raise RuntimeError("Training produced no batches; reduce --batch-size")
            stream.flush()
            save_grid(ema, fixed_noise, fixed_labels, args.output / f"samples_epoch_{epoch:03d}.png")
            save_checkpoint(args.output / "checkpoint.pt", generator, discriminator,
                            ema, optim_g, optim_d, scaler, epoch, step, config)
            print(f"epoch {epoch} complete: D={d_total/batches:.3f} "
                  f"G={g_total/batches:.3f}; elapsed={(time.time()-begin)/3600:.2f} h", flush=True)
            if (args.max_steps and step >= args.max_steps or
                    time.time() - begin >= args.max_hours * 3600):
                print("Stopped after a saved epoch; resume from checkpoint.pt if needed.", flush=True)
                break


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/full-wikiart-gan/evaluate.py
"""Evaluate the EMA generator with held-out real paintings using Ignite IS/FID."""

import argparse
import json
from pathlib import Path

import torch
from ignite.metrics import FID, InceptionScore
from ignite.metrics.gan.utils import InceptionModel
from torch.utils.data import DataLoader, Subset
from torchvision.transforms.functional import resize

from model import Generator
from train import make_dataset


def inception_input(images: torch.Tensor) -> torch.Tensor:
    return resize(((images + 1) / 2).clamp(0, 1), [299, 299], antialias=True)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--max-images", type=int, default=1000)
    parser.add_argument("--batch-size", type=int, default=16)
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    state = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    config = state["config"]
    generator = Generator(len(config["classes"]), config["image_size"],
                          config["latent_dim"], config["base_channels"]).to(device)
    generator.load_state_dict(state["ema"])
    generator.eval()
    full, _, holdout = make_dataset(args.data, config["seed"], config["image_size"],
                                    min_images=100 if config["allow_small_data"] else 50000)
    if full.classes != config["classes"] or len(full) != config["dataset_size"]:
        raise ValueError("Evaluation dataset does not match training checkpoint")
    count = min(args.max_images, len(holdout))
    if count < 100:
        raise ValueError("At least 100 held-out images are required")
    loader = DataLoader(Subset(holdout, range(count)), batch_size=args.batch_size,
                        shuffle=False, num_workers=0)
    fid = FID(num_features=2048,
              feature_extractor=InceptionModel(return_features=True, device=device),
              device=device)
    fid_blog = FID(device=device)
    inception_score = InceptionScore(device=device)
    for metric in (fid, fid_blog, inception_score):
        metric.reset()
    torch.manual_seed(config["seed"] + 1000)
    with torch.inference_mode():
        for i, (real, labels) in enumerate(loader, 1):
            real = real.to(device)
            labels = labels.to(device)
            noise = torch.randn(real.size(0), config["latent_dim"], device=device)
            fake = generator(noise, labels)
            real_for_metric = inception_input(real)
            fake_for_metric = inception_input(fake)
            fid.update((fake_for_metric, real_for_metric))
            fid_blog.update((fake_for_metric, real_for_metric))
            inception_score.update(fake_for_metric)
            print(f"Evaluated {min(i * args.batch_size, count)}/{count} pairs", flush=True)
    result = {"fid_pool3": float(fid.compute()),
              "fid_ignite_blog_default": float(fid_blog.compute()),
              "inception_score": float(inception_score.compute()),
              "real_images": count, "generated_images": count,
              "checkpoint_epoch": state["epoch"], "training_images": config["train_size"],
              "heldout_images": config["holdout_size"], "source_images": config["dataset_size"],
              "image_size": config["image_size"], "styles": len(config["classes"]),
              "generator": "EMA", "metric_library": "pytorch-ignite",
              "resize": "bilinear antialias to 299x299",
              "real_split": "seeded held-out 10%; matching style labels for fakes"}
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/full-wikiart-gan/sample.py
"""Generate style-conditioned painting grids from a full WikiArt checkpoint."""

import argparse
from pathlib import Path

import torch
from torchvision.utils import save_image

from model import Generator


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--style", type=str, help="One of the style names in config.json")
    parser.add_argument("--count", type=int, default=64)
    parser.add_argument("--seed", type=int, default=1234)
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    state = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    config = state["config"]
    generator = Generator(len(config["classes"]), config["image_size"],
                          config["latent_dim"], config["base_channels"]).to(device)
    generator.load_state_dict(state["ema"])
    generator.eval()
    torch.manual_seed(args.seed)
    if args.style:
        if args.style not in config["classes"]:
            parser.error(f"Unknown style {args.style!r}; choose from {config['classes']}")
        labels = torch.full((args.count,), config["classes"].index(args.style),
                            device=device, dtype=torch.long)
    else:
        styles = torch.linspace(0, len(config["classes"]) - 1, 8,
                                device=device).long()
        labels = styles.repeat_interleave((args.count + 7) // 8)[:args.count]
    with torch.inference_mode():
        images = generator(torch.randn(args.count, config["latent_dim"], device=device), labels)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    save_image(images.float().cpu(), args.output, nrow=8,
               normalize=True, value_range=(-1, 1))
    print(f"Saved {args.count} paintings to {args.output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/full-wikiart-gan/plot_losses.py
"""Plot raw and smoothed losses saved by train.py."""

import argparse
import csv
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def smooth(values, window=25):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    kernel = np.ones(window)
    return np.convolve(values, kernel, mode="same") / np.convolve(np.ones_like(values), kernel, mode="same")


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--losses", type=Path, required=True)
    p.add_argument("--output", type=Path, required=True)
    args = p.parse_args()
    with args.losses.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    steps = [int(row["step"]) for row in rows]
    fig, ax = plt.subplots(figsize=(8, 3.5), dpi=180)
    for key, color, label in (("loss_d", "#1e6091", "Discriminator"),
                              ("loss_g", "#c45d39", "Generator")):
        values = np.array([float(row[key]) for row in rows])
        ax.plot(steps, values, color=color, alpha=0.18, linewidth=0.7)
        ax.plot(steps, smooth(values), color=color, linewidth=1.7, label=label)
    ax.set(xlabel="Training step", ylabel="Hinge objective")
    ax.grid(alpha=0.16)
    ax.legend(frameon=False)
    fig.tight_layout()
    args.output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(args.output)


if __name__ == "__main__":
    main()


## Check the GPU and metric dependency

In [ ]:
import importlib.util
import torch

print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Session options.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA kernel test:', torch.ones(1, device='cuda'))
if importlib.util.find_spec('ignite') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps',
                    'pytorch-ignite==0.5.4'], check=True)
from torchvision.models import Inception_V3_Weights
Inception_V3_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True)
print('Inception-v3 evaluation weights are ready.')


## Prepare the entire attached dataset

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'prepare.py'),
                '--input-root', '/kaggle/input', '--output', str(DATA),
                '--size', '128'], check=True)
manifest = json.loads((DATA / 'full_provenance.json').read_text())
shutil.copy2(DATA / 'full_provenance.json', PROJECT / 'full_provenance.json')
print('Source files:', manifest['available_images'])
print('Prepared files:', manifest['prepared_images'])
print('Styles:', len(manifest['styles']))
print('Unreadable files skipped:', len(manifest['skipped']))
if manifest['prepared_images'] < 50000 or len(manifest['styles']) != 27:
    raise RuntimeError('The attached dataset appears incomplete; expected at least '
                       '50,000 images in 27 WikiArt style folders.')


## Train the full-data model

Set `RESUME_CHECKPOINT` to the path of a checkpoint from a previous saved
notebook version if you need another session. Attach that notebook version via
**Add Input**, expand it, and copy the path ending in `run/checkpoint.pt`.
Leave it as `None` for a new run. Each session stops after about eight training
hours, after saving the current epoch. `EPOCHS=40` is a target, not a promise
that one Kaggle session will finish it.


In [ ]:
EPOCHS = 40
RESUME_CHECKPOINT = None  # Example: '/kaggle/input/my-previous-run/run/checkpoint.pt'

command = [sys.executable, str(PROJECT / 'train.py'),
           '--data', str(DATA), '--output', str(RUN),
           '--image-size', '128', '--epochs', str(EPOCHS),
           '--batch-size', '64', '--base-channels', '32',
           '--workers', '4', '--max-hours', '8']
if RESUME_CHECKPOINT is not None:
    checkpoint_path = Path(RESUME_CHECKPOINT)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    previous_run = checkpoint_path.parent
    for name in ('losses.csv', 'real_grid.png'):
        if (previous_run / name).is_file() and not (RUN / name).is_file():
            shutil.copy2(previous_run / name, RUN / name)
    command.extend(['--resume', str(checkpoint_path)])
subprocess.run(command, check=True)
state = torch.load(RUN / 'checkpoint.pt', map_location='cpu', weights_only=True)
print(f"Finished epoch {state['epoch']} of {EPOCHS}.")
if state['epoch'] < EPOCHS:
    print('Training is partial. Save this version, then attach its output and '
          'set RESUME_CHECKPOINT in a new session to continue.')


## Evaluate the current checkpoint and save figures

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'evaluate.py'),
                '--checkpoint', str(RUN / 'checkpoint.pt'),
                '--data', str(DATA),
                '--output', str(RUN / 'metrics.json'),
                '--max-images', '1000', '--batch-size', '16'], check=True)
subprocess.run([sys.executable, str(PROJECT / 'plot_losses.py'),
                '--losses', str(RUN / 'losses.csv'),
                '--output', str(RUN / 'loss_curves.png')], check=True)
subprocess.run([sys.executable, str(PROJECT / 'sample.py'),
                '--checkpoint', str(RUN / 'checkpoint.pt'),
                '--output', str(RUN / 'generated_grid.png'),
                '--count', '64'], check=True)
metrics = json.loads((RUN / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))


## View and package the results

In [ ]:
from IPython.display import Image as NotebookImage, display

display(NotebookImage(filename=str(RUN / 'real_grid.png')))
display(NotebookImage(filename=str(RUN / 'loss_curves.png')))
display(NotebookImage(filename=str(RUN / 'generated_grid.png')))

archive_base = str(PROJECT / 'full_run_artifacts')
archive = shutil.make_archive(archive_base, 'zip', root_dir=PROJECT,
                              base_dir='run')
print('Download archive:', archive)
print('Provenance:', PROJECT / 'full_provenance.json')
print('Completed epochs:', metrics['checkpoint_epoch'], '/', EPOCHS)


### What to send back

Download `full_run_artifacts.zip` and `full_provenance.json` from the saved
notebook version. The archive includes `run/checkpoint.pt`, metrics, losses,
loss curves, and generated examples. If the recorded epoch is below 40, the
run is still partial; use that checkpoint to resume. A lower FID than the old
run is desirable, but the full-data result needs to be measured before any
claim about improvement or grading.
